In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import dataretrieval.waterdata as waterdata
import os
import dataretrieval.utils as utils

In [ ]:
# Looks like waterdata only includes the most recent data. We need NWIS for historical data (training)
from dataretrieval import nwis

In [ ]:
import truststore
truststore.inject_into_ssl()

In [ ]:
def nwis_grabber(site_id, param, start_date, end_date):

    df, metadata = nwis.get_dv(
        site=site_id,
        parameterCd=param,
        start=start_date,
        end=end_date
    )

    if df.empty:
        print(f"Warning: No data found for {site_id}")
        return df, metadata

    name_map = {
        '63680': 'Turbidity',
        '00010': 'Temp_C',
        '00400': 'pH',
        '00095': 'Specific_Cond',
        '00300': 'Dissolved_Oxygen',
        '00060': 'Streamflow'
    }

    df = df.reset_index()

    # identify only data columns (exclude _cd flags)
    value_cols = [c for c in df.columns if c != 'site_no' and not c.endswith('_cd')]

    df_long = df.melt(
        id_vars=['datetime', 'site_no'],
        value_vars=value_cols,
        var_name='temp_name',
        value_name='value'
    )

    df_long[['parameter', 'stat_label']] = df_long['temp_name'].str.rsplit('_', n=1, expand=True)

    moment_labels = ['Maximum', 'Minimum', 'Mean', 'Median']
    df_final = df_long[df_long['stat_label'].isin(moment_labels)].copy()

    # map stats to cleaner names
    df_final['stat_label'] = df_final['stat_label'].replace({
        'Maximum': 'Max',
        'Minimum': 'Min'
    })

    # human-readable column names
    df_final['param_name'] = df_final['parameter'].map(name_map) + "_" + df_final['stat_label']

    # pivot back wide
    df_pivoted = df_final.pivot(
        index=['datetime', 'site_no'],
        columns='param_name',
        values='value'
    ).reset_index()

    df_pivoted = df_pivoted.rename(columns={'datetime': 'Date'})
    df_pivoted['Date'] = pd.to_datetime(df_pivoted['Date']).dt.tz_localize(None)
    df_pivoted.columns.name = None

    print(f"Success! Grabbed data for {site_id}. Shape: {df_pivoted.shape}")

    return df_pivoted, metadata

In [ ]:
# Pulling all params from sonde above Strontia Reservoir
params = ['63680', '00010', '00400', '00095', '00300']
historical_sp, metadata = nwis_grabber(site_id='06707525', param=params, start_date='2022-04-01', end_date='2026-08-24')

In [ ]:
historical_sp.tail()

In [ ]:
historical_sp.to_csv(r"C:\Users\jslawson\OneDrive - Denver Water\SCO\Source_water_early_warning_systems\Data\USGS_South_Platte.csv", index = False)